In [1]:
println(scala.util.Properties.versionNumberString)

2.13.17


In [2]:
import $ivy.`org.apache.spark::spark-sql:4.1.2`

import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .config("spark.log.level", "WARN")
  .master("local[*]")
  .appName("scala-notebook")
  .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

println(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/13 21:01:13 INFO SparkContext: Running Spark version 4.1.2
26/07/13 21:01:13 INFO SparkContext: OS info Linux, 7.1.3-200.fc44.x86_64, amd64
26/07/13 21:01:13 INFO SparkContext: Java version 21.0.11+10-1-24.04.2-Ubuntu
26/07/13 21:01:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting Spark log level to "WARN".


4.1.2


import $ivy.$
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@5d4fa5a4

In [3]:
spark.range(10).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



In [4]:
import almond.display.Html
import org.apache.spark.sql.DataFrame

object DataFrameHtml {

  private def escapeHtml(value: String): String =
    value.flatMap {
      case '&'  => "&amp;"
      case '<'  => "&lt;"
      case '>'  => "&gt;"
      case '"'  => "&quot;"
      case '\'' => "&#39;"
      case c    => c.toString
    }

  implicit class DataFrameHtmlOps(private val df: DataFrame) {

    def showHtml(
        limit: Int = 20,
        truncate: Int = 100
    )  = {

      require(limit > 0, "limit must be greater than zero")

      val rows = df.take(limit)

      def formatValue(value: Any): String = {
        val text =
          Option(value)
            .map(_.toString)
            .getOrElse("null")

        val shortened =
          if (truncate > 0 && text.length > truncate)
            text.take(truncate) + "…"
          else
            text

        escapeHtml(shortened)
      }

      val headers =
        df.schema.fields
          .map { field =>
            val name = escapeHtml(field.name)
            val dataType = escapeHtml(field.dataType.simpleString)

            s"""
               |<th>
               |  <div>$name</div>
               |  <small>$dataType</small>
               |</th>
               |""".stripMargin
          }
          .mkString

      val body =
        rows.map { row =>
          val cells =
            (0 until row.length)
              .map { index =>
                s"<td>${formatValue(row.get(index))}</td>"
              }
              .mkString

          s"<tr>$cells</tr>"
        }.mkString

      val html =
        s"""
           |<div class="spark-dataframe-container">
           |  <table class="spark-dataframe">
           |    <thead>
           |      <tr>$headers</tr>
           |    </thead>
           |
           |    <tbody>
           |      $body
           |    </tbody>
           |  </table>
           |
           |  <div class="spark-dataframe-summary">
           |    Showing up to $limit rows
           |  </div>
           |</div>
           |
           |<style>
           |  .spark-dataframe-container {
           |    max-width: 100%;
           |    overflow-x: auto;
           |  }
           |
           |  .spark-dataframe {
           |    min-width: 100%;
           |    border-collapse: collapse;
           |    font-family: monospace;
           |    font-size: 14px;
           |  }
           |
           |  .spark-dataframe th {
           |    padding: 8px 12px;
           |    border: 1px solid #666;
           |    text-align: left;
           |    white-space: nowrap;
           |    background: rgba(127, 127, 127, 0.20);
           |  }
           |
           |  .spark-dataframe th small {
           |    opacity: 0.65;
           |    font-weight: normal;
           |  }
           |
           |  .spark-dataframe td {
           |    padding: 6px 12px;
           |    border: 1px solid #666;
           |    white-space: nowrap;
           |  }
           |
           |  .spark-dataframe tbody tr:hover {
           |    background: rgba(127, 127, 127, 0.15);
           |  }
           |
           |  .spark-dataframe-summary {
           |    margin-top: 6px;
           |    opacity: 0.7;
           |  }
           |</style>
           |""".stripMargin

      Html(html)
    }
  }
}

import DataFrameHtml._

import almond.display.Html
import org.apache.spark.sql.DataFrame
defined object DataFrameHtml
import DataFrameHtml._

In [5]:
spark.range(10).toDF().show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



In [6]:
spark.range(10).toDF().showHtml()

id bigint
0
1
2
3
4
5
6
7
8
9


In [7]:
import DataFrameHtml._

val employeeData = Seq(
  (1, "Alice", 32, "Data Engineer", 12500.50, true),
  (2, "Bob", 41, "Data Architect", 15800.00, true),
  (3, "Carol", 27, "Data Analyst", 9200.75, false),
  (4, "David", 36, "Platform Engineer", 13750.25, true)
)

val employees = spark
  .createDataFrame(employeeData)
  .toDF(
    "id",
    "name",
    "age",
    "job_title",
    "monthly_salary",
    "active"
  )

employees.showHtml()

id int,name string,age int,job_title string,monthly_salary double,active boolean
1,Alice,32,Data Engineer,12500.5,true
2,Bob,41,Data Architect,15800.0,true
3,Carol,27,Data Analyst,9200.75,false
4,David,36,Platform Engineer,13750.25,true


import DataFrameHtml._
employeeData: Seq[(Int, String, Int, String, Double, Boolean)] = List(
  (1, "Alice", 32, "Data Engineer", 12500.5, true),
  (2, "Bob", 41, "Data Architect", 15800.0, true),
  (3, "Carol", 27, "Data Analyst", 9200.75, false),
  (4, "David", 36, "Platform Engineer", 13750.25, true)
)
employees: DataFrame = [id: int, name: string ... 4 more fields]

In [8]:
employees.showHtml(
  limit = 3,
  truncate = 30
)

id int,name string,age int,job_title string,monthly_salary double,active boolean
1,Alice,32,Data Engineer,12500.5,true
2,Bob,41,Data Architect,15800.0,true
3,Carol,27,Data Analyst,9200.75,false
